 Installing Required Libraries


In [ ]:

!pip install transformers datasets pandas torch

Import Libraries

In [ ]:
import transformers
import torch
import pandas as pd
print("Libraries installed successfully!")

Load Email(Enron) Dataset

In [ ]:
!wget https://www.cs.cmu.edu/~./enron/enron_mail_20150507.tar.gz

Extract Dataset

In [ ]:
!tar -xzf enron_mail_20150507.tar.gz

In [ ]:
import os

os.listdir("maildir")[:10]

Extract Email Body and Subject

In [ ]:
import os
import pandas as pd

emails = []

for root, dirs, files in os.walk("maildir"):
    for file in files:
        path = os.path.join(root, file)
        try:
            with open(path, "r", errors="ignore") as f:
                text = f.read()

            if "Subject:" in text:
                subject = text.split("Subject:")[1].split("\n")[0]
                body = text.split("\n\n",1)[1] if "\n\n" in text else ""

                emails.append((body, subject))
        except:
            pass

df = pd.DataFrame(emails, columns=["email_body","subject"])

df.head()

In [ ]:
df = df.sample(5000, random_state=42)

print(len(df))
df.head()


Prepare Input and Target Text

In [ ]:
df = df.rename(columns={
    "email_body":"input_text",
    "subject":"target_text"
})

df["input_text"] = "generate subject: " + df["input_text"]

df.head()


 Convert Data to HuggingFace Dataset

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset

Load T5 Tokenizer and Model

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Model loaded")

Tokenization

In [ ]:

def tokenize(example):

    input_enc = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    target_enc = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

    input_enc["labels"] = target_enc["input_ids"]

    return input_enc


tokenized_dataset = dataset.map(tokenize)

Model Training


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

In [ ]:
import torch
email = "Dear professor, I have completed my machine learning assignment and attached the report and code files. Please review them."
input_text = "generate subject: " + email
inputs = tokenizer.encode(input_text, return_tensors="pt")
inputs = inputs.to(model.device)
outputs = model.generate(
    inputs,
    max_length=20,
    num_beams=5,
    early_stopping=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
!pip install nltk

In [ ]:
email = "Please attend the project meeting tomorrow at 10 AM"

input_text = "generate subject: " + email

inputs = tokenizer.encode(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(inputs, max_length=20)

generated_subject = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Subject:", generated_subject)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

actual_subject = "Project Meeting Tomorrow"

reference = [actual_subject.lower().split()]
candidate = generated_subject.lower().split()

smooth = SmoothingFunction().method1

bleu = sentence_bleu(reference, candidate, smoothing_function=smooth)

print("BLEU Score:", bleu)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

actual_subject = "Project Meeting Tomorrow"
generated_subject = generated_subject
true_tokens = actual_subject.lower().split()
pred_tokens = generated_subject.lower().split()

all_words = list(set(true_tokens + pred_tokens))

true_vector = [1 if word in true_tokens else 0 for word in all_words]
pred_vector = [1 if word in pred_tokens else 0 for word in all_words]
precision = precision_score(true_vector, pred_vector)
recall = recall_score(true_vector, pred_vector)
f1 = f1_score(true_vector, pred_vector)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)